In [ ]:
# import libraries
try:
  # %tensorflow_version only exists in Colab.
  !pip install tf-nightly
except Exception:
  pass
import tensorflow as tf
import pandas as pd
from tensorflow import keras
!pip install tensorflow-datasets
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
# Load and prepare the SMS datasets.
SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

train_df = pd.read_csv(
    train_file_path,
    sep='\t',
    header=None,
    names=['label', 'message']
)
test_df = pd.read_csv(
    test_file_path,
    sep='\t',
    header=None,
    names=['label', 'message']
)

label_map = {'ham': 0, 'spam': 1}
train_labels = train_df['label'].map(label_map).astype('float32').to_numpy()
test_labels = test_df['label'].map(label_map).astype('float32').to_numpy()
train_texts = train_df['message'].astype(str).to_numpy()
test_texts = test_df['message'].astype(str).to_numpy()

print(f'Training examples: {len(train_texts)}')
print(f'Validation examples: {len(test_texts)}')
print(train_df['label'].value_counts())


In [ ]:
# Build and train a neural-network text classifier.
# TF-IDF with unigrams + bigrams works well for the short SMS messages
# while the dense layers learn a nonlinear spam/ham decision boundary.
MAX_TOKENS = 20000

vectorizer = keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    standardize='lower_and_strip_punctuation',
    split='whitespace',
    ngrams=2,
    output_mode='tf_idf'
)
vectorizer.adapt(train_texts)

model = keras.Sequential([
    keras.Input(shape=(), dtype=tf.string),
    vectorizer,
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.30),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dropout(0.20),
    keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    train_texts,
    train_labels,
    epochs=20,
    batch_size=32,
    validation_split=0.20,
    callbacks=[early_stopping],
    verbose=1
)

test_loss, test_accuracy, test_auc = model.evaluate(
    test_texts, test_labels, verbose=0
)
print(f'Validation accuracy: {test_accuracy:.4f}')
print(f'Validation AUC: {test_auc:.4f}')


In [ ]:
# function to predict messages based on model
# (should return list containing prediction and label, ex. [0.008318834938108921, 'ham'])
def predict_message(pred_text):
  probability = float(
      model.predict(np.array([pred_text]), verbose=0).reshape(-1)[0]
  )
  label = 'spam' if probability >= 0.5 else 'ham'
  prediction = [probability, label]
  return prediction

pred_text = "how are you doing today?"

prediction = predict_message(pred_text)
print(prediction)

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
